In [1]:
import sounddevice as sd
import numpy as np
import librosa
import joblib
from tensorflow.keras.models import load_model

In [7]:
# 1. 모델 & 스케일러 불러오기
model = load_model('audio_emotion_model.keras')
scaler = joblib.load('scaler.save')

# 2. 감정 라벨 
emotion_labels = ['angry', 'happy', 'neutral', 'sad']

# 3. 오디오 녹음 함수
def record_audio(duration=3, sr=22050):
    print(f"\n 말해주세요...")
    audio = sd.rec(int(duration * sr), samplerate=sr, channels=1)
    sd.wait()
    audio = audio.flatten()
    return audio, sr

# 4. MFCC 추출 + 스케일링
def extract_features(audio, sr):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=100)
    mfcc_mean = np.mean(mfcc.T, axis=0)
    mfcc_scaled = scaler.transform([mfcc_mean])
    mfcc_scaled = np.expand_dims(mfcc_scaled, axis=-1)  
    return mfcc_scaled

# 5. 예측 함수
def predict_emotion(audio, sr):
    features = extract_features(audio, sr)         # shape: (100,) 또는 (100,1)

    # 항상 (100, 1) 형태로 만들기
    features = np.reshape(features, (100, 1))       # (100,1)
    features = np.expand_dims(features, axis=0)     # (1,100,1) - 배치 차원 추가

    # 예측
    pred = model.predict(features)
    emotion_idx = np.argmax(pred)
    confidence = np.max(pred)
    return emotion_labels[emotion_idx], confidence


# 6. 실행
if __name__ == "__main__":
    while True:
        audio, sr = record_audio()
        emotion, confidence = predict_emotion(audio, sr)
        print(f"예측된 감정: {emotion.upper()} (신뢰도: {confidence:.2%})")

        again = input("\n계속하려면 Enter, 종료하려면 q: ")
        if again.lower() == 'q':
            break



 말해주세요...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
예측된 감정: NEUTRAL (신뢰도: 99.99%)



계속하려면 Enter, 종료하려면 q:  q
